In [125]:
import pandas as pd
import numpy as np

**Генератор признаков**

In [126]:
def generate_features(n_samples, n_features, prefix):
    df = pd.DataFrame()
    
    nominal = [1000, 500, 333, 777]
    ordinal = [1, 2, 3, 4]
    
    for i in range(1, n_features + 1):
        if i % 4 == 1:
            df[f'{prefix}_bin_{i}'] = np.random.choice([1, 0], n_samples)
        elif i % 4 == 2:
            df[f'{prefix}_nominal_{i}'] = np.random.choice(nominal, n_samples)
        elif i % 4 == 3:
            df[f'{prefix}_ordinal_{i}'] = np.random.choice(ordinal, n_samples)
        else:
            df[f'{prefix}_quantitative_{i}'] = np.random.uniform(10, 1000, n_samples)
            
    return df


**Функция коллизии**

In [127]:
def collision(row1, row2):
    diff_sum = 0
    v1 = row1.values
    v2 = row2.values
    cols = row1.index 
    
    for i in range(len(cols)):
        col_name = cols[i]
        val1 = v1[i]
        val2 = v2[i]
        
        if 'bin' in col_name or 'nominal' in col_name:
            diff_sum += 1 if val1 != val2 else 0
        elif 'ordinal' in col_name:
            if val1 == 777 or val2 == 777:
                diff_sum += 2
            elif val1 == 333 or val2 == 333:
                diff_sum += 1
        else:
            diff_sum += abs(val1 - val2) / 1000
    
    return 1 if diff_sum < len(cols) * 0.4 else 0

**Генерация датасетов**

In [128]:
def create_dataset(n_samples, n_features_obj1, n_features_obj2):
    obj1_df = generate_features(n_samples, n_features_obj1, 'Obj1')
    obj2_df = generate_features(n_samples, n_features_obj2, 'Obj2')
    
    dataset = pd.concat([obj1_df, obj2_df], axis=1)
    
    results = []
    for i in range(len(dataset)):
        res = collision(obj1_df.iloc[i], obj2_df.iloc[i])
        results.append(res)

    dataset['collision'] = results
    
    return dataset


In [129]:
sample_sizes = [50, 250, 750, 1500]
feature_counts = [5, 9, 15]
datasets = {}
dataset_id = 1

for n_samples in sample_sizes:
    for n_features in feature_counts:
        df = create_dataset(n_samples, n_features, n_features)
        name = f"ID{dataset_id}_S{n_samples}_F{n_features}"
        datasets[name] = df
        dataset_id += 1
        df.to_csv(name)

for ds in datasets:
    print(ds)

datasets["ID3_S50_F15"].head()

ID1_S50_F5
ID2_S50_F9
ID3_S50_F15
ID4_S250_F5
ID5_S250_F9
ID6_S250_F15
ID7_S750_F5
ID8_S750_F9
ID9_S750_F15
ID10_S1500_F5
ID11_S1500_F9
ID12_S1500_F15


,Obj1_bin_1,Obj1_nominal_2,Obj1_ordinal_3,Obj1_quantitative_4,Obj1_bin_5,Obj1_nominal_6,Obj1_ordinal_7,Obj1_quantitative_8,Obj1_bin_9,Obj1_nominal_10,...,Obj2_ordinal_7,Obj2_quantitative_8,Obj2_bin_9,Obj2_nominal_10,Obj2_ordinal_11,Obj2_quantitative_12,Obj2_bin_13,Obj2_nominal_14,Obj2_ordinal_15,collision
0,0,333,3,727.205461,0,777,2,698.748100,0,333,...,4,671.710327,0,777,4,586.450697,0,1000,1,0
1,0,333,3,20.342675,0,777,2,869.124786,0,333,...,4,708.795889,1,500,2,836.463086,0,777,3,0
2,0,1000,3,286.346725,1,777,4,739.960897,1,777,...,2,362.326903,1,777,1,767.700208,0,1000,1,1
3,0,777,3,475.715018,0,500,4,108.981496,0,333,...,4,324.438998,1,333,3,287.822763,1,500,1,1
4,0,777,2,332.156348,1,1000,4,627.971007,1,1000,...,4,287.487610,1,777,2,710.799042,0,1000,4,1


In [130]:
from sklearn.metrics import f1_score
from sklearn.base import clone
from sklearn.model_selection import train_test_split

def evaluate_model_on_datasets(model, datasets):
    model_name = model.__class__.__name__
    results = {'Method': model_name}
    
    for i, df in enumerate(datasets):
        y = df["collision"]
        X = df.drop(["collision"], axis=1)
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=228)
        
        model.fit(X_train, y_train)
        
        y_pred = model.predict(X_test)
        score = f1_score(y_test, y_pred, average='weighted')
        
        results[f'Dataset_{i+1}'] = score
        
    return pd.Series(results)

In [131]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

In [132]:
models = [
    KNeighborsClassifier(),
    GaussianNB(),
    RandomForestClassifier(random_state=42),
    LogisticRegression(max_iter=1000)
]

rows = []
for model in models:
    row = evaluate_model_on_datasets(model, list(datasets.values()))
    rows.append(row)

results_df = pd.DataFrame(rows)
results_df['average'] = results_df.mean(axis=1, numeric_only=True)
results_df

c:\Users\Slava\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\Slava\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://s

,Method,Dataset_1,Dataset_2,Dataset_3,Dataset_4,Dataset_5,Dataset_6,Dataset_7,Dataset_8,Dataset_9,Dataset_10,Dataset_11,Dataset_12,average
0,KNeighborsClassifier,0.600000,0.800000,0.515152,0.673071,0.492929,0.447214,0.675599,0.481389,0.562290,0.645333,0.598750,0.579851,0.589298
1,GaussianNB,0.616667,0.400000,0.709890,0.607619,0.429091,0.572413,0.418205,0.469474,0.544643,0.479588,0.470906,0.522925,0.520118
2,RandomForestClassifier,0.600000,0.400000,0.709890,0.713015,0.528176,0.595513,0.693598,0.652078,0.546586,0.872788,0.641438,0.629683,0.631897
3,LogisticRegression,0.516484,0.484848,0.709890,0.568381,0.492929,0.603205,0.418205,0.457430,0.465812,0.457551,0.438239,0.486119,0.508258


In [133]:
from sklearn.model_selection import GridSearchCV

def evaluate_tuned_model_on_datasets(model, datasets, param_grid):
    model_name = model.__class__.__name__
    results = {'Method': model_name}
    
    for i, df in enumerate(datasets):
        y = df["collision"]
        X = df.drop(["collision"], axis=1)
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=228)
        
        grid_search = GridSearchCV(
            estimator=model,
            param_grid=param_grid,
            scoring='f1_weighted',
            cv=5,
            n_jobs=-1
        )
        
        grid_search.fit(X_train, y_train)
        
        best_model = grid_search.best_estimator_
        y_pred = best_model.predict(X_test)
        
        score = f1_score(y_test, y_pred, average='weighted')
        results[f'Dataset_{i+1}'] = score
        
    return pd.Series(results)

In [134]:
param_grids = [
    {
        'n_neighbors': [3, 5, 7, 9, 11, 13, 15, 17],
        'weights': ['uniform', 'distance'],
        'metric': ['euclidean', 'manhattan', 'minkowski'],
        'p': [1, 2, 3]
    },
    {
        'var_smoothing': [1e-10, 1e-9, 1e-8, 1e-7]
    },
    {
        'n_estimators': [100, 200, 300],
        'max_depth': [None, 5, 10, 20],
        'min_samples_split': [2, 5, 10]
    },
    {
        'C': [0.001, 0.01, 0.1, 1.0, 10.0],
        'solver': ['lbfgs', 'liblinear', 'newton-cg', 'saga'],
    }
]

tuned_rows = []

# for model, grid, df in zip(models, param_grids, list(datasets.values())):
#     grid_search = GridSearchCV(model, grid, cv=min(7, len(df)//20), scoring='f1_weighted', n_jobs=-1)
#     row = evaluate_model_on_datasets(grid_search, list(datasets.values()))
#     tuned_rows.append(row)
tuned_rows = []

for model, grid in zip(models, param_grids):
    result_series = evaluate_tuned_model_on_datasets(model, list(datasets.values()), grid)
    tuned_rows.append(result_series)

tuned_results_df = pd.DataFrame(tuned_rows)
tuned_results_df['average'] = tuned_results_df.mean(axis=1, numeric_only=True)
tuned_results_df

c:\Users\Slava\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\Slava\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://s

,Method,Dataset_1,Dataset_2,Dataset_3,Dataset_4,Dataset_5,Dataset_6,Dataset_7,Dataset_8,Dataset_9,Dataset_10,Dataset_11,Dataset_12,average
0,KNeighborsClassifier,0.515152,0.800000,0.515152,0.673071,0.492929,0.523846,0.643717,0.506667,0.558036,0.626599,0.592608,0.586685,0.586205
1,GaussianNB,0.616667,0.400000,0.709890,0.607619,0.429091,0.572413,0.418205,0.469474,0.544643,0.479588,0.470906,0.522925,0.520118
2,RandomForestClassifier,0.762500,0.400000,0.709890,0.704035,0.562811,0.624119,0.696722,0.662696,0.559687,0.866295,0.665625,0.630012,0.653699
3,LogisticRegression,0.516484,0.484848,0.709091,0.552258,0.504236,0.546505,0.418205,0.451354,0.505876,0.457551,0.438239,0.499822,0.507039


In [135]:
results_df

,Method,Dataset_1,Dataset_2,Dataset_3,Dataset_4,Dataset_5,Dataset_6,Dataset_7,Dataset_8,Dataset_9,Dataset_10,Dataset_11,Dataset_12,average
0,KNeighborsClassifier,0.600000,0.800000,0.515152,0.673071,0.492929,0.447214,0.675599,0.481389,0.562290,0.645333,0.598750,0.579851,0.589298
1,GaussianNB,0.616667,0.400000,0.709890,0.607619,0.429091,0.572413,0.418205,0.469474,0.544643,0.479588,0.470906,0.522925,0.520118
2,RandomForestClassifier,0.600000,0.400000,0.709890,0.713015,0.528176,0.595513,0.693598,0.652078,0.546586,0.872788,0.641438,0.629683,0.631897
3,LogisticRegression,0.516484,0.484848,0.709890,0.568381,0.492929,0.603205,0.418205,0.457430,0.465812,0.457551,0.438239,0.486119,0.508258
